# Common Failure Analysis: 4-Class RMM Classification

This notebook analyzes clips that **all 3 models** (V-JEPA2, PoseC3D, and Qwen2.5-VL) failed to classify correctly in the 4-class RMM task.

**Goal**: Identify systematic failure patterns that could inform:
- Data quality improvements
- Model architecture choices
- Annotation review priorities

**Date**: December 23, 2024


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

print("Libraries loaded successfully!")


## 1. Load Predictions from All 3 Models

Loading cross-validation predictions from:
- **V-JEPA2**: RGB-based video encoder with SAM3 person cropping
- **PoseC3D**: Skeleton-based action recognition using COCO keypoints
- **Qwen2.5-VL**: Zero-shot vision-language model


In [ ]:
# Define paths. Notebooks have no __file__, so walk up from the kernel's cwd to
# find the repo root (the directory holding paths.py). Works whether the kernel
# starts in this folder or at the repo root.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())

VJEPA_DIR = REPO_ROOT / "v-jepa/runs/vjepa2_rmm_cv/f64_lr1e-5_bs1_acc8_ep20_crop_4cls"
POSEC3D_DIR = REPO_ROOT / "pyskl/work_dirs/posec3d/cv/4class_conf04"
QWEN_FILE = REPO_ROOT / "insights/vlm/qwen_outputs/cv_4class_7489124/frames16_samples4/cv_cv_4class/predictions_clip_all_folds.csv"

# Load V-JEPA predictions (all folds)
vjepa_dfs = []
for fold in [0, 1, 2]:
    path = VJEPA_DIR / f"fold_{fold}/clip_level_preds.csv"
    df = pd.read_csv(path)
    df['fold'] = fold
    df['vjepa_correct'] = df['label_name'] == df['pred_name']
    vjepa_dfs.append(df)
vjepa = pd.concat(vjepa_dfs, ignore_index=True)
vjepa = vjepa.rename(columns={'segment_id': 'id', 'label_name': 'label', 'pred_name': 'vjepa_pred'})

# Load PoseC3D predictions (all folds)
posec3d_dfs = []
for fold in [0, 1, 2]:
    path = POSEC3D_DIR / f"fold{fold}/eval_val/predictions_clip.csv"
    df = pd.read_csv(path)
    df['fold'] = fold
    df['posec3d_correct'] = df['true_class'] == df['pred_class']
    posec3d_dfs.append(df)
posec3d = pd.concat(posec3d_dfs, ignore_index=True)
posec3d = posec3d.rename(columns={'segment_id': 'id', 'pred_class': 'posec3d_pred'})

# Load Qwen predictions
qwen = pd.read_csv(QWEN_FILE)
qwen['qwen_correct'] = qwen['correct']
qwen = qwen.rename(columns={'prediction': 'qwen_pred'})

print(f"V-JEPA clips: {len(vjepa)}")
print(f"PoseC3D clips: {len(posec3d)}")
print(f"Qwen clips: {len(qwen)}")


In [ ]:
# Load clip lengths from pickle files
import pickle

PKL_DIR = BASE_DIR / "actreg/pyskl/data/sails/cv/4class_conf04"

clip_lengths = {}
for fold in [0, 1, 2]:
    with open(PKL_DIR / f"fold{fold}.pkl", 'rb') as f:
        data = pickle.load(f)
    for ann in data['annotations']:
        segment_id = ann['frame_dir']
        clip_lengths[segment_id] = ann['total_frames']

print(f"Loaded clip lengths for {len(clip_lengths)} clips")
print(f"Length range: {min(clip_lengths.values())} - {max(clip_lengths.values())} frames")
print(f"Mean length: {np.mean(list(clip_lengths.values())):.1f} frames")


In [ ]:
# Merge all predictions on clip id
merged = vjepa[['id', 'label', 'vjepa_pred', 'vjepa_correct', 'video_id', 
                'n_children', 'n_adults', 'quality_bucket', 'mixed_video', 
                'crop_applied', 'crop_reason', 'pred_conf']].merge(
    posec3d[['id', 'posec3d_pred', 'posec3d_correct']], on='id', how='inner'
).merge(
    qwen[['id', 'qwen_pred', 'qwen_correct']], on='id', how='inner'
)

# Extract age group from video_id
merged['age_group'] = merged['video_id'].str.extract(r'_(\d+)_month_')[0].fillna('unknown')

# Add clip length from pickle files
merged['clip_length'] = merged['id'].map(clip_lengths)

print(f"Total clips with predictions from all 3 models: {len(merged)}")
print(f"Clips with length info: {merged['clip_length'].notna().sum()}")
print(f"\nError rates:")
print(f"  V-JEPA errors:  {(~merged['vjepa_correct']).sum():3d} ({(~merged['vjepa_correct']).mean()*100:.1f}%)")
print(f"  PoseC3D errors: {(~merged['posec3d_correct']).sum():3d} ({(~merged['posec3d_correct']).mean()*100:.1f}%)")
print(f"  Qwen errors:    {(~merged['qwen_correct']).sum():3d} ({(~merged['qwen_correct']).mean()*100:.1f}%)")


## 2. Identify Common Failures

Finding clips where **all 3 models** predicted incorrectly — these represent the most challenging cases in the dataset.


In [ ]:
# Find clips where all 3 models failed
all_failed = merged[
    (merged['vjepa_correct'] == False) & 
    (merged['posec3d_correct'] == False) & 
    (merged['qwen_correct'] == False)
].copy()

# Also find clips where V-JEPA + PoseC3D both failed (the two best models)
vjepa_posec3d_failed = merged[
    (merged['vjepa_correct'] == False) & 
    (merged['posec3d_correct'] == False)
].copy()

print(f"{'='*60}")
print(f"CLIPS WHERE ALL 3 MODELS FAILED: {len(all_failed)} ({len(all_failed)/len(merged)*100:.1f}% of dataset)")
print(f"{'='*60}")
print(f"\nBreakdown by TRUE label:")
print(all_failed['label'].value_counts().to_frame('count'))
print(f"\n{'='*60}")
print(f"CLIPS WHERE V-JEPA + POSEC3D BOTH FAILED: {len(vjepa_posec3d_failed)}")
print(f"{'='*60}")


## 3. Visualize Class Distribution of Failures


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Class distribution of all-3-failed clips vs dataset
class_order = ['hands flapping', 'jumping', 'rocking', 'spinning']
colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']

# Dataset distribution
dataset_dist = merged['label'].value_counts()[class_order]
failed_dist = all_failed['label'].value_counts().reindex(class_order, fill_value=0)

x = np.arange(len(class_order))
width = 0.35

bars1 = axes[0].bar(x - width/2, dataset_dist.values, width, label='Full Dataset', color=colors, alpha=0.5)
bars2 = axes[0].bar(x + width/2, failed_dist.values, width, label='All 3 Failed', color=colors, edgecolor='black', linewidth=2)

axes[0].set_ylabel('Number of Clips', fontsize=12)
axes[0].set_title('Class Distribution: Dataset vs Common Failures', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.replace(' ', '\n') for c in class_order])
axes[0].legend()

# Plot 2: Failure rate by class
failure_rate = (failed_dist / dataset_dist * 100).fillna(0)
bars = axes[1].bar(class_order, failure_rate.values, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Failure Rate (%)', fontsize=12)
axes[1].set_title('Common Failure Rate by Class', fontsize=14, fontweight='bold')
axes[1].set_xticklabels([c.replace(' ', '\n') for c in class_order])

# Add percentage labels on bars
for bar, rate in zip(bars, failure_rate.values):
    axes[1].annotate(f'{rate:.1f}%', 
                     xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('class_failure_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Key Insight: Rocking has the highest common failure rate, followed by jumping.")


## 4. Confusion Patterns

What do the models predict when they're all wrong? This reveals systematic biases.


In [ ]:
# Analyze confusion patterns for V-JEPA + PoseC3D joint failures
confusion = vjepa_posec3d_failed.groupby(['label', 'vjepa_pred', 'posec3d_pred']).size().reset_index(name='count')
confusion = confusion.sort_values('count', ascending=False)

print("Top 15 Confusion Patterns (True → V-JEPA | PoseC3D):\n")
print(confusion.head(15).to_string(index=False))

# Create a heatmap for model agreement on failures
print("\n" + "="*60)
print("When V-JEPA and PoseC3D BOTH fail, do they agree on the wrong prediction?")
print("="*60)

vjepa_posec3d_failed['models_agree'] = vjepa_posec3d_failed['vjepa_pred'] == vjepa_posec3d_failed['posec3d_pred']
agreement = vjepa_posec3d_failed['models_agree'].value_counts()
print(f"\nModels agree on wrong prediction: {agreement.get(True, 0)} clips ({agreement.get(True, 0)/len(vjepa_posec3d_failed)*100:.1f}%)")
print(f"Models disagree (different wrong predictions): {agreement.get(False, 0)} clips ({agreement.get(False, 0)/len(vjepa_posec3d_failed)*100:.1f}%)")


In [ ]:
# Visualize what failed clips are confused as
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# V-JEPA confusion for failed clips
vjepa_confusion = vjepa_posec3d_failed.groupby(['label', 'vjepa_pred']).size().unstack(fill_value=0)
vjepa_confusion = vjepa_confusion.reindex(index=class_order, columns=class_order, fill_value=0)
sns.heatmap(vjepa_confusion, annot=True, fmt='d', cmap='Reds', ax=axes[0], 
            xticklabels=[c.split()[0] for c in class_order],
            yticklabels=[c.split()[0] for c in class_order])
axes[0].set_title('V-JEPA Predictions on Failed Clips', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True Label')

# PoseC3D confusion for failed clips
posec3d_confusion = vjepa_posec3d_failed.groupby(['label', 'posec3d_pred']).size().unstack(fill_value=0)
posec3d_confusion = posec3d_confusion.reindex(index=class_order, columns=class_order, fill_value=0)
sns.heatmap(posec3d_confusion, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=[c.split()[0] for c in class_order],
            yticklabels=[c.split()[0] for c in class_order])
axes[1].set_title('PoseC3D Predictions on Failed Clips', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_on_failures.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Key Insight: Both models over-predict 'hands flapping' for jumping and rocking clips.")


## 5. Metadata Analysis of Failed Clips

Are there patterns in video quality, age group, or scene complexity that predict common failures?


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

# 1. Quality bucket comparison
quality_all = merged['quality_bucket'].value_counts(normalize=True) * 100
quality_failed = all_failed['quality_bucket'].value_counts(normalize=True) * 100

quality_df = pd.DataFrame({'Dataset': quality_all, 'Common Failures': quality_failed}).fillna(0)
quality_df.plot(kind='bar', ax=axes[0, 0], color=['steelblue', 'coral'], edgecolor='black')
axes[0, 0].set_title('Quality Bucket Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Percentage (%)')
axes[0, 0].tick_params(axis='x', rotation=0)
axes[0, 0].legend()

# 2. Age group comparison
age_all = merged['age_group'].value_counts(normalize=True) * 100
age_failed = all_failed['age_group'].value_counts(normalize=True) * 100

age_df = pd.DataFrame({'Dataset': age_all, 'Common Failures': age_failed}).fillna(0)
age_order = [a for a in ['14', '36', 'unknown'] if a in age_df.index]
age_df = age_df.loc[age_order]
age_df.plot(kind='bar', ax=axes[0, 1], color=['steelblue', 'coral'], edgecolor='black')
axes[0, 1].set_title('Age Group Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Percentage (%)')
axes[0, 1].set_xlabel('Age (months)')
axes[0, 1].tick_params(axis='x', rotation=0)
axes[0, 1].legend()

# 3. Number of children
children_all = merged['n_children'].value_counts(normalize=True) * 100
children_failed = all_failed['n_children'].value_counts(normalize=True) * 100

children_df = pd.DataFrame({'Dataset': children_all, 'Common Failures': children_failed}).fillna(0).sort_index()
children_df.plot(kind='bar', ax=axes[1, 0], color=['steelblue', 'coral'], edgecolor='black')
axes[1, 0].set_title('Number of Children in Frame', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Percentage (%)')
axes[1, 0].set_xlabel('# Children')
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].legend()

# 4. Mixed video (boolean) - more robust than crop_applied
mixed_all = merged['mixed_video'].value_counts(normalize=True) * 100
mixed_failed = all_failed['mixed_video'].value_counts(normalize=True) * 100

mixed_df = pd.DataFrame({'Dataset': mixed_all, 'Common Failures': mixed_failed}).fillna(0)
mixed_df = mixed_df.reindex([False, True], fill_value=0)
mixed_df.plot(kind='bar', ax=axes[1, 1], color=['steelblue', 'coral'], edgecolor='black')
axes[1, 1].set_title('Mixed Video (Multi-person scene)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Percentage (%)')
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(['Single Person', 'Mixed'], rotation=0)
axes[1, 1].legend()

# 5. Clip length distribution (histogram)
merged_lengths = merged['clip_length'].dropna()
failed_lengths = all_failed['clip_length'].dropna()

axes[2, 0].hist(merged_lengths, bins=30, alpha=0.6, label='Dataset', color='steelblue', edgecolor='black')
axes[2, 0].hist(failed_lengths, bins=30, alpha=0.7, label='Common Failures', color='coral', edgecolor='black')
axes[2, 0].axvline(merged_lengths.mean(), color='steelblue', linestyle='--', linewidth=2, label=f'Dataset mean: {merged_lengths.mean():.0f}')
axes[2, 0].axvline(failed_lengths.mean(), color='coral', linestyle='--', linewidth=2, label=f'Failures mean: {failed_lengths.mean():.0f}')
axes[2, 0].set_title('Clip Length Distribution (frames)', fontsize=12, fontweight='bold')
axes[2, 0].set_xlabel('Clip Length (frames)')
axes[2, 0].set_ylabel('Count')
axes[2, 0].legend(fontsize=9)

# 6. Clip length by failure status (box plot)
length_data = [merged_lengths.values, failed_lengths.values]
bp = axes[2, 1].boxplot(length_data, labels=['Dataset', 'Common Failures'], patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('coral')
bp['boxes'][1].set_alpha(0.7)
axes[2, 1].set_title('Clip Length: Dataset vs Common Failures', fontsize=12, fontweight='bold')
axes[2, 1].set_ylabel('Clip Length (frames)')

plt.tight_layout()
plt.savefig('metadata_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Statistical summary
print("="*60)
print("METADATA SUMMARY: COMMON FAILURES vs DATASET")
print("="*60)

print("\n📊 Quality Bucket (low quality):")
print(f"  Dataset:  {quality_all.get('low', 0):.1f}%")
print(f"  Failures: {quality_failed.get('low', 0):.1f}%")

print("\n📊 Age Group (36 months - older children):")
print(f"  Dataset:  {age_all.get('36', 0):.1f}%")
print(f"  Failures: {age_failed.get('36', 0):.1f}%")

print("\n📊 Multi-child scenes (n_children > 1):")
multi_child_dataset = (merged['n_children'] > 1).mean() * 100
multi_child_failed = (all_failed['n_children'] > 1).mean() * 100
print(f"  Dataset:  {multi_child_dataset:.1f}%")
print(f"  Failures: {multi_child_failed:.1f}%")

print("\n📊 Mixed videos (multiple people in scene):")
mixed_dataset = merged['mixed_video'].mean() * 100
mixed_failed = all_failed['mixed_video'].mean() * 100
print(f"  Dataset:  {mixed_dataset:.1f}%")
print(f"  Failures: {mixed_failed:.1f}%")

print("\n📊 Clip Length (frames):")
dataset_len = merged['clip_length'].dropna()
failed_len = all_failed['clip_length'].dropna()
print(f"  Dataset mean:   {dataset_len.mean():.1f} frames (std: {dataset_len.std():.1f})")
print(f"  Failures mean:  {failed_len.mean():.1f} frames (std: {failed_len.std():.1f})")
print(f"  Dataset median:  {dataset_len.median():.0f} frames")
print(f"  Failures median: {failed_len.median():.0f} frames")

# Check for short clips in failures
short_threshold = 60  # 2 seconds at 30fps
short_dataset = (dataset_len < short_threshold).mean() * 100
short_failed = (failed_len < short_threshold).mean() * 100
print(f"\n  Short clips (<{short_threshold} frames / 2 sec):")
print(f"    Dataset:  {short_dataset:.1f}%")
print(f"    Failures: {short_failed:.1f}%")

# Compute odds ratio for key factors
print("\n📊 Failure Rate Analysis:")
for label in class_order:
    total = (merged['label'] == label).sum()
    failed = (all_failed['label'] == label).sum()
    rate = failed / total * 100 if total > 0 else 0
    print(f"  {label}: {failed}/{total} failed ({rate:.1f}%)")


## 6. Videos with Multiple Failed Clips

These videos are particularly problematic — consider reviewing annotations or video quality.


In [ ]:
# Find videos with multiple clips where both V-JEPA and PoseC3D failed
video_failures = vjepa_posec3d_failed.groupby('video_id').agg({
    'id': 'count',
    'label': 'first',
    'quality_bucket': 'first',
    'n_children': 'first',
    'age_group': 'first'
}).rename(columns={'id': 'failed_clips'})

multi_fail = video_failures[video_failures['failed_clips'] > 1].sort_values('failed_clips', ascending=False)

print(f"Videos with MULTIPLE clips where V-JEPA + PoseC3D both failed:\n")
print(multi_fail.to_string())

print(f"\n📋 Total: {len(multi_fail)} videos with 2+ commonly failed clips")
print(f"   These videos account for {multi_fail['failed_clips'].sum()} of {len(vjepa_posec3d_failed)} joint-failure clips")


In [ ]:
# Get original video paths for the 10 videos with 2+ commonly failed clips
import json

# Load video metadata with original paths
VIDEO_META_PATH = BASE_DIR / "actreg/dataprep/video_meta.json"
with open(VIDEO_META_PATH) as f:
    video_meta = json.load(f)

# Build lookup: mask_cache_path -> original_path
cache_to_original = {}
for record in video_meta['records']:
    if record.get('mask_cache_path') and record.get('original_path'):
        cache_to_original[record['mask_cache_path']] = record['original_path']

# Get the crop_cache_path for failed videos from the vjepa data
failed_video_ids = multi_fail.index.tolist()

# Find unique crop_cache_paths for each failed video
video_to_original = {}
for vid in failed_video_ids:
    # Get the first clip from this video to find its cache path
    clip_rows = vjepa[vjepa['video_id'] == vid]
    if len(clip_rows) > 0:
        cache_path = clip_rows.iloc[0].get('crop_cache_path', None)
        if cache_path and cache_path in cache_to_original:
            video_to_original[vid] = cache_to_original[cache_path]
        else:
            video_to_original[vid] = f"[Cache path not found: {cache_path}]"

# Display the results
print("="*80)
print("ORIGINAL VIDEO PATHS FOR TOP 10 MULTI-FAILURE VIDEOS")
print("="*80)
print()

for i, (vid, row) in enumerate(multi_fail.iterrows(), 1):
    original_path = video_to_original.get(vid, "Path not found")
    print(f"{i:2d}. {vid}")
    print(f"    Failed clips: {row['failed_clips']}")
    print(f"    True label: {row['label']}")
    print(f"    Original: {original_path}")
    print()

# Also save to a text file for easy copy-paste
with open('multi_failure_video_paths.txt', 'w') as f:
    f.write("# Videos with 2+ commonly failed clips (V-JEPA + PoseC3D both wrong)\n")
    f.write("# Format: video_id | failed_clips | label | original_path\n\n")
    for vid, row in multi_fail.iterrows():
        original_path = video_to_original.get(vid, "Path not found")
        f.write(f"{vid}\t{row['failed_clips']}\t{row['label']}\t{original_path}\n")

print("✅ Saved to multi_failure_video_paths.txt")


## 7. Complete List of Common Failures

These 48 clips represent the hardest cases — all 3 models failed on them.


In [ ]:
# Display all clips where all 3 models failed
display_cols = ['id', 'label', 'vjepa_pred', 'posec3d_pred', 'qwen_pred', 
                'quality_bucket', 'n_children', 'age_group']

all_failed_display = all_failed[display_cols].copy()
all_failed_display = all_failed_display.sort_values(['label', 'id'])

print(f"All {len(all_failed)} clips where V-JEPA, PoseC3D, AND Qwen all failed:\n")
pd.set_option('display.max_rows', 100)
display(all_failed_display)


## 8. Conclusions & Recommendations

### Key Findings

1. **48 clips (7.7%)** are common failures across all 3 models
2. **Rocking is the hardest class** — highest common failure rate
3. **Jumping clips are often confused with hands flapping** — likely due to co-occurring arm movements
4. **36-month age group is over-represented** in failures
5. **Multi-child scenes** are disproportionately difficult

### Recommendations

| Priority | Action | Rationale |
|----------|--------|-----------|
| 🔴 High | Review annotations for rocking clips | Highest failure rate, possible label ambiguity |
| 🔴 High | Review the 10 multi-failure videos | Concentrated failure points |
| 🟡 Medium | Add jumping-specific data augmentation | Jumping/flapping confusion is systematic |
| 🟡 Medium | Improve multi-child scene handling | Higher failure rate when n_children > 1 |
| 🟢 Low | Consider hierarchical classification | Separate gross-motor (jumping/spinning) vs fine-motor (flapping/rocking) |


In [ ]:
# Save failed clips to CSV for review
all_failed[display_cols + ['video_id']].to_csv('common_failures_all_3_models.csv', index=False)
vjepa_posec3d_failed[display_cols + ['video_id']].to_csv('common_failures_vjepa_posec3d.csv', index=False)

print("✅ Saved outputs:")
print("   - common_failures_all_3_models.csv (48 clips)")
print("   - common_failures_vjepa_posec3d.csv (68 clips)")
print("   - class_failure_distribution.png")
print("   - confusion_on_failures.png")
print("   - metadata_analysis.png")
